# pyQuil Tutorial: The Quil Compiler

*Condensed from the [pyQuil docs, v4.17](https://pyquil-docs.rigetti.com/en/stable/compiler.html).*

## Expectations for program contents

Real QPUs support a much smaller native gate set than pyQuil offers. The Quil SDK includes an optimizing compiler, **quilc**, that translates arbitrary Quil to native Quil, and native Quil to executables suitable for Rigetti hardware.

> **Prerequisites:** `pip install pyquil`. Cells that run programs need the `qvm` and `quilc` servers running (the setup cell below starts them). Cells marked `# Requires QCS access` need a Rigetti Quantum Cloud Services account and will fail on a local-only setup.

## Setup

Use the **Python (Rigetti)** kernel (see the README for one-time kernel registration). Run the next cell once per session to start the `quilc` and `qvm` servers in the background (`!cmd &` doesn't work in Jupyter, so this uses `subprocess.Popen`).

In [1]:
#Finding running process. If the server is running, skip the next shell.
!ps -u $USER -o pid,ppid,etime,comm,cmd | grep -E "[q]vm|[q]uilc"

In [2]:
import subprocess
ACT = "source /ocean/projects/pscstaff/nnepal/Rigetti/.venv/bin/activate"
for cmd in ("qvm -S", "quilc -S -P"):
    name = cmd.split()[0]
    subprocess.Popen(["bash", "-c", f"{ACT} && exec {cmd}"],
                     stdout=open(f"{name}.log", "w"), stderr=subprocess.STDOUT,
                     start_new_session=True)

## Interacting with the compiler

### Setup

After installing the SDK, start a local compiler server with `quilc -S`. A `QuantumComputer` from `get_qc()` connects to it; use `.compile()`:

In [6]:
from pyquil.quil import Pragma, Program
from pyquil.api import get_qc
from pyquil.gates import CNOT, H

qc = get_qc("9q-square-qvm")

executable = qc.compile(Program(H(0), CNOT(0, 1), CNOT(1, 2)))

print(executable)

RZ(pi) 0
RX(pi/2) 0
RZ(pi/2) 0
RX(-pi/2) 0
RX(pi/2) 1
RZ(pi/2) 1
RX(-pi/2) 1
CZ 1 0
RZ(pi) 1
RX(pi/2) 1
RZ(pi/2) 1
RX(-pi/2) 1
RZ(-pi) 1
RX(pi/2) 2
RZ(pi/2) 2
RX(-pi/2) 2
CZ 2 1
RZ(pi) 0
RZ(pi) 1
RZ(pi) 2
RX(pi/2) 2
RZ(pi/2) 2
RX(-pi/2) 2
RZ(-pi) 2



Expected output (native gates only — `RZ`, `RX(±π/2)`, `CZ`):

```
RZ(pi/2) 0
RX(pi/2) 0
RZ(-pi/2) 1
RX(pi/2) 1
CZ 1 0
RX(-pi/2) 1
RZ(-pi/2) 2
RX(pi/2) 2
CZ 2 1
RZ(-pi/2) 0
RZ(-pi/2) 1
RX(-pi/2) 2
RZ(pi/2) 2
```

> **Note:** For a QVM, printing the executable shows readable native Quil. For a live QPU target the output is encrypted, because the program has been translated for the hardware.

### The compiler interface

The compiler is at `qc.compiler` and follows the `AbstractCompiler` interface:

- **`compiler.quil_to_native_quil(program, *, protoquil)`** — converts Quil to native Quil for the ISA. `protoquil=True` restricts input/output to *protoquil* (code executable on a QPU). Returns program metadata when the server was started with `-P`.
- **`compiler.native_quil_to_executable(nq_program)`** — converts native Quil into an executable for a QVM or QPU.

`qc.compile()` does both. The equivalent two-step version:

In [7]:
from pyquil.quil import Pragma, Program
from pyquil.api import get_qc
from pyquil.gates import CNOT, H

qc = get_qc("9q-square-qvm")

p = Program(H(0), CNOT(0, 1), CNOT(1, 2))

native_prog = qc.compiler.quil_to_native_quil(p, protoquil=True)  # the docs call this `np`
print(native_prog)

exe = qc.compiler.native_quil_to_executable(native_prog)
print("executable:\n")
print(exe)

RZ(pi) 0
RX(pi/2) 0
RZ(pi/2) 0
RX(-pi/2) 0
RX(pi/2) 1
RZ(pi/2) 1
RX(-pi/2) 1
CZ 1 0
RZ(pi) 1
RX(pi/2) 1
RZ(pi/2) 1
RX(-pi/2) 1
RZ(-pi) 1
RX(pi/2) 2
RZ(pi/2) 2
RX(-pi/2) 2
CZ 2 1
RZ(pi) 0
RZ(pi) 1
RZ(pi) 2
RX(pi/2) 2
RZ(pi/2) 2
RX(-pi/2) 2
RZ(-pi) 2

executable:

RZ(pi) 0
RX(pi/2) 0
RZ(pi/2) 0
RX(-pi/2) 0
RX(pi/2) 1
RZ(pi/2) 1
RX(-pi/2) 1
CZ 1 0
RZ(pi) 1
RX(pi/2) 1
RZ(pi/2) 1
RX(-pi/2) 1
RZ(-pi) 1
RX(pi/2) 2
RZ(pi/2) 2
RX(-pi/2) 2
CZ 2 1
RZ(pi) 0
RZ(pi) 1
RZ(pi) 2
RX(pi/2) 2
RZ(pi/2) 2
RX(-pi/2) 2
RZ(-pi) 2



### Timeouts

Complex circuits may need more compilation time:

In [8]:
qc = get_qc("2q-qvm", compiler_timeout=100)  # 100 seconds

### Legal compiler input

Programs for a QPU must have this structure:

1. an optional initial `RESET`
2. a block of native gates: `RZ(θ)` (any θ), `RX(k·π/2)` (integer k), or `CZ q0 q1` on neighboring qubits; some devices also support `XY(θ) q0 q1`
3. a block of `MEASURE` instructions

Use `protoquil=True` to guarantee QPU-compatible output:

```python
compiler.quil_to_native_quil(program, protoquil=True)
qc.compile(program, protoquil=True)
```

> **Note:** If the server was started with `-P`, `protoquil=False` overrides it; `protoquil=None` defers to the server's setting.

### Compilation metadata

When the compiler runs with `-P`, `quil_to_native_quil()` also returns statistics:

| Field | Meaning |
|---|---|
| `final_rewiring` | Logical-to-physical qubit mapping |
| `gate_depth` | Longest sequence of instructions sharing resources |
| `multiqubit_gate_depth` | Gate depth counting only multi-qubit gates |
| `gate_volume` | Total gate count |
| `program_duration` | Duration with gates executed in parallel |
| `qpu_runtime_estimation` | Estimated runtime in ms (for parametric programs, multiply by the parameter-space size) |
| `program_fidelity` | Estimated fidelity of the compiled program |
| `topological_swaps` | Number of topological swaps inserted |

In [9]:
from pyquil import get_qc, Program

qc = get_qc("8q-qvm")

p = Program("RX(pi) 0")

native_p = qc.compiler.quil_to_native_quil(p, protoquil=True)

print(native_p)
print(native_p.native_quil_metadata.qpu_runtime_estimation)

RX(pi) 0

183.51910400390625


## Region-specific compiler features through `PRAGMA`

### Preserved regions

`PRAGMA PRESERVE_BLOCK` … `PRAGMA END_PRESERVE_BLOCK` stops the compiler from modifying the enclosed instructions.

> **Warning:** If a preserved block isn't legal QPU input, it isn't guaranteed to execute.

Example: prepare a Bell state, wait, then read out:

In [11]:
preserved = Program("""
DECLARE ro BIT[2]

#   prepare a Bell state
H 0
CNOT 0 1

#   wait a while
PRAGMA PRESERVE_BLOCK
I 0
I 1
I 0
I 1
I 0
I 1
PRAGMA END_PRESERVE_BLOCK

#   and read out the results
MEASURE 0 ro[0]
MEASURE 1 ro[1]
""")
print(get_qc("9q-square-qvm").compile(preserved))

DECLARE ro BIT[2]
RZ(pi) 0
RX(pi/2) 0
RZ(pi/2) 0
RX(-pi/2) 0
RX(pi/2) 1
RZ(pi/2) 1
RX(-pi/2) 1
CZ 1 0
RZ(pi) 0
RZ(pi) 1
RX(pi/2) 1
RZ(pi/2) 1
RX(-pi/2) 1
RZ(-pi) 1
I 0
I 1
I 0
I 1
I 0
I 1
MEASURE 1 ro[1]
MEASURE 0 ro[0]



### Parallelizable regions

Mark sequences of commuting gates with `PRAGMA COMMUTING_BLOCKS` / `PRAGMA END_COMMUTING_BLOCKS`, and each individual sequence with `PRAGMA BLOCK` / `PRAGMA END_BLOCK`. The compiler may then reorder them.

> **Warning:** Marking blocks as commuting when they don't commute produces incorrect results.

A VQE-style example:

In [10]:
vqe_style = Program("""
DECLARE ro BIT[4]

# Stage one
H 0
H 1
H 2
H 3

# Stage two
PRAGMA COMMUTING_BLOCKS
PRAGMA BLOCK
CNOT 0 1
RZ(0.4) 1
CNOT 0 1
PRAGMA END_BLOCK
PRAGMA BLOCK
CNOT 1 2
RZ(0.6) 2
CNOT 1 2
PRAGMA END_BLOCK
PRAGMA BLOCK
CNOT 2 3
RZ(0.8) 3
CNOT 2 3
PRAGMA END_BLOCK
PRAGMA BLOCK
CNOT 0 3
RZ(0.9) 3
CNOT 0 3
PRAGMA END_BLOCK
PRAGMA END_COMMUTING_BLOCKS

# Stage three
H 0
H 1
H 2
H 3

MEASURE 0 ro[0]
MEASURE 1 ro[1]
MEASURE 2 ro[2]
MEASURE 3 ro[3]
""")
print(get_qc("9q-square-qvm").compile(vqe_style))

DECLARE ro BIT[4]
RZ(-0.9199372448290238) 6
RX(pi/2) 6
RZ(pi) 6
RX(-pi/2) 6
RZ(2.2216554087607694) 6
RZ(pi) 7
RX(pi/2) 7
RZ(pi/2) 7
RX(-pi/2) 7
CZ 6 7
RZ(pi/2) 6
RX(pi/2) 6
RZ(0.7999999999999999) 6
RX(-pi/2) 6
RZ(-pi/2) 6
RZ(pi) 7
CZ 6 7
RX(pi/2) 5
RZ(pi/2) 5
RX(-pi/2) 5
RZ(4.3399276606314245) 5
RZ(-pi/2) 8
RX(pi/2) 8
RZ(1.9432576465481604) 8
RX(-pi/2) 8
RZ(-pi/2) 8
XY(pi) 5 8
RX(pi/2) 5
RZ(0.39999999999999947) 5
RX(-pi/2) 5
CZ 5 8
RZ(-pi/2) 7
RX(pi/2) 7
RZ(pi/2) 7
RX(-pi/2) 7
RZ(pi/2) 7
RZ(1.4038387663875815) 8
RX(pi/2) 8
RZ(pi) 8
RX(-pi/2) 8
RZ(-2.110215206955478) 8
CZ 7 8
RX(pi/2) 7
RZ(0.5999999999999996) 7
RX(-pi/2) 7
CZ 7 8
RZ(2.1588064212647846) 5
RX(pi/2) 5
RZ(2.3587440414043943) 5
RX(-pi/2) 5
RZ(-0.34824739083712497) 5
RZ(-0.694045646079439) 8
RX(pi/2) 8
RZ(2.1979513283399337) 8
RX(-pi/2) 8
RZ(-0.15599041579132505) 8
XY(pi) 8 5
CZ 8 5
RZ(2.618844230094729) 7
RX(pi/2) 7
RZ(2.549808709516152) 7
RX(-pi/2) 7
RZ(1.287686064345597) 7
RZ(-1.9393321434360735) 8
RX(pi/2) 8
RZ(0.48033630

## Rewirings

When a multi-qubit instruction uses a qubit pair that isn't connected on the device, the compiler rearranges qubits. Comments like `# Entering rewiring: #(n0 n1 ... nk)` mean logical qubit `j` now lives on physical qubit `nj`.

### SWAPs

The compiler inserts `SWAP` gates to move instructions onto neighboring qubits. Initial swaps can be *virtual* — they only change the initial rewiring and don't appear in the output.

Example: non-neighboring qubits on a 3-qubit line `0 – 1 – 2`:

In [12]:
import networkx as nx
from pyquil import Program, get_qc
from pyquil.api import QCSClient
from pyquil.api._quantum_computer import _get_qvm_with_topology
from pyquil.gates import CZ

graph = nx.from_edgelist([(0, 1), (1, 2)])
qc = _get_qvm_with_topology(
    client_configuration=QCSClient(),
    name="line",
    topology=graph,
    noisy=False,
    qvm_type="qvm",
    compiler_timeout=30.0,
    execution_timeout=30.0,
)

p = Program(CZ(0, 2))
print(qc.compile(p))

CZ 2 1



Expected output: `CZ 2 1` then `HALT` — the swap was virtual.

A more complex program that needs real SWAPs:

In [13]:
from pyquil.gates import H, CZ

p = Program(CZ(0, 1), H(0), CZ(1, 2), CZ(0, 2))
print(qc.compile(p))

CZ 2 1
RZ(-0.9199372448290238) 2
RX(pi/2) 2
RZ(pi) 2
RX(-pi/2) 2
RZ(2.2216554087607694) 2
CZ 2 1
CZ 1 0
RZ(pi/2) 1
RZ(-1.8398744896580477) 2
RX(pi/2) 2
RZ(pi) 2
RX(-pi/2) 2
RZ(-0.2690781628631509) 2
XY(pi) 1 2
RZ(pi) 1
RX(pi/2) 1
RZ(pi/2) 1
RX(-pi/2) 1
CZ 1 0



> **Note:** A SWAP costs three CZ or three XY gates. Devices that support both can build a SWAP from two gates (one CZ and one XY).

### Initial rewiring

Control how the initial mapping is built with `PRAGMA INITIAL_REWIRING`:

```
# <type> can be NAIVE, RANDOM, PARTIAL, or GREEDY
#
# The double quotes are required.
PRAGMA INITIAL_REWIRING "<type>"
```

### The default initial rewiring strategy

Without `INITIAL_REWIRING`, the compiler picks:

- **`NAIVE`** when every instruction's qubits already satisfy the device topology
- **`PARTIAL`** otherwise

The docs illustrate these strategies on a real device topology (`Aspen-X`), which requires QCS access. The cells below use it as in the docs; the final cell reproduces the idea locally on the 3-qubit line.

In [14]:
# Requires QCS access. Neighboring qubits -> NAIVE is applied, output: CZ 3 4
from pyquil import Program, get_qc
from pyquil.gates import CZ

qc_aspen = get_qc("Aspen-X", as_qvm=True)
p = Program(CZ(3, 4))
print(qc_aspen.compile(p))

GetISAError: Call failed during http request: error in refresh_qcs_token: No refresh token is configured within the selected QCS credential.

#### NAIVE

Logical qubit `i` maps directly to physical qubit `i`. The compiler generally will **not** move an instruction's qubits, even if that results in poor fidelity.

In [15]:
# Requires QCS access. A low-fidelity CZ is kept as-is: output CZ 0 1
p = Program('PRAGMA INITIAL_REWIRING "NAIVE"', CZ(0, 1))
print(qc_aspen.compile(p))

NameError: name 'qc_aspen' is not defined

In [ ]:
# Requires QCS access. Non-neighboring qubits need a virtual swap: docs output CZ 6 5
p = Program('PRAGMA INITIAL_REWIRING "NAIVE"', CZ(0, 2))
print(qc_aspen.compile(p))

> **Note:** When NAIVE mode has to rewire, the result isn't guaranteed to have optimal fidelity.

#### PARTIAL

The compiler starts with an empty logical-to-physical mapping and fills it in as it compiles, giving it the freedom to choose a mapping that optimizes the program's fidelity.

In [ ]:
# Requires QCS access. The compiler moves the CZ to a better pair: docs output CZ 20 27
p = Program('PRAGMA INITIAL_REWIRING "PARTIAL"', CZ(0, 1))
print(qc_aspen.compile(p))

#### GREEDY

The compiler builds the initial mapping from a greedy optimization of distances between the program's qubits and the device's. It's faster than PARTIAL but gives lower-fidelity programs — use it when compile speed matters more than fidelity.

#### Which strategy should I use?

| Desired effect | Recommended strategy |
|---|---|
| Maximize program execution fidelity | `PARTIAL` |
| Keep the input program's qubits where possible | `NAIVE` |
| Faster qubit allocation at the expense of fidelity | `GREEDY` |

> **Note:** Each strategy has the drawbacks described above.

### Try it locally: compare strategies on the line topology

In [16]:
for strategy in ["NAIVE", "PARTIAL", "GREEDY"]:
    p = Program(f'PRAGMA INITIAL_REWIRING "{strategy}"', CZ(0, 2))
    print(f"--- {strategy} ---")
    print(qc.compile(p))

--- NAIVE ---
PRAGMA INITIAL_REWIRING "NAIVE"
CZ 1 2

--- PARTIAL ---
PRAGMA INITIAL_REWIRING "PARTIAL"
CZ 2 1

--- GREEDY ---
PRAGMA INITIAL_REWIRING "GREEDY"
CZ 0 1



## Common error messages

- **`! ! ! Error: Matrices do not lie in the same projective class.`** The compiler decomposed an operator into native instructions, but the result doesn't match the original. Usually the matrix isn't unitary or a `DEFGATE` block is invalid. Rarely it's a floating-point precision issue, and recompiling may fix it. If not, contact support@rigetti.com or file a GitHub issue.

In [2]:
!kill -9 80794
!kill -9 80795